# Preprocessing des Données Kickstarter

## Informations sur le dataset
Le dataset vient des données scrappées par Web Robots sur la plateforme Kickstarter, entre Juillet 2025 et Juin 2026.
Chaque mois est composé de plusieurs fichiers CSV qu'il conviendra d'abord de fusionner afin d'évaluer convenablement la volumétrie du dataset.

### Nombre de fichiers CSV par mois
* **Juillet 2025** : 81
* **Août 2025** : 83
* **Septembre 2025** : 83
* **Octobre 2025** : 83
* **Novembre 2025** : 83
* **Décembre 2025** : 84
* **Janvier 2026** : 84
* **Février 2026** : 85
* **Mars 2026** : 85
* **Avril 2026** : 86
* **Mai 2026** : 86
* **Juin 2026** : 86

En voyant cette liste, on comprend que le robot de crawl récupère **toutes les données** Kickstarter une fois par mois. Il ne s'agit pas d'une capture des projets qui ont vu le jour chaque mois, mais bien d'une capture totale des données Kickstarter.

Pour le vérifier, nous allons comparer un projet vu en juillet 2025, et vu en juin 2026.

Inspectons d'abord les informations du mois de juin 2026 :

In [1]:
import pandas as pd

raw_202606 = pd.read_csv('/Users/mariamalaborde/Documents/DataScientest/memoire-2026/data/1-merged/202606-kickstarter.csv', sep=',', encoding='utf-8')
raw_202507 = pd.read_csv('/Users/mariamalaborde/Documents/DataScientest/memoire-2026/data/1-merged/202507-kickstarter.csv', sep=',', encoding='utf-8')

In [2]:
raw_202606.isna().sum()

backers_count                              0
blurb                                    122
category                                   0
converted_pledged_amount               20249
country                                    0
country_displayable_name                   0
created_at                                 0
creator                                    0
currency                                   0
currency_symbol                            0
currency_trailing_code                     0
current_currency                           0
deadline                                   0
disable_communication                      0
fx_rate                                    0
goal                                       0
id                                         0
is_disliked                           272518
is_in_post_campaign_pledging_phase         0
is_launched                                0
is_liked                              272518
is_starrable                               0
launched_a

In [3]:
raw_202606['id'].duplicated().sum()

np.int64(61712)

In [4]:
# Conversion des timestamps unix en datetime

cols_dates = ['deadline', 'launched_at', 'created_at']

for col in cols_dates:
    if col in raw_202507.columns:
        raw_202507[col] = pd.to_datetime(raw_202507[col], unit='s', errors='coerce')

print(raw_202507[cols_dates].dtypes)

for col in cols_dates:
    if col in raw_202606.columns:
        raw_202606[col] = pd.to_datetime(raw_202606[col], unit='s', errors='coerce')

print(raw_202606[cols_dates].dtypes)

# Localisation du projet le plus ancien
projet_plus_ancien = raw_202507.loc[raw_202507['launched_at'].idxmin()]

print("=== PROJET LE PLUS ANCIEN DU DATASET ===")
print(f"ID          : {projet_plus_ancien['id']}")
print(f"Nom         : {projet_plus_ancien['name']}")
print(f"Date Launch : {projet_plus_ancien['launched_at']}")
print(f"Statut      : {projet_plus_ancien['state']}")

deadline       datetime64[s]
launched_at    datetime64[s]
created_at     datetime64[s]
dtype: object
deadline       datetime64[s]
launched_at    datetime64[s]
created_at     datetime64[s]
dtype: object
=== PROJET LE PLUS ANCIEN DU DATASET ===
ID          : 1557383515
Nom         : AeveQuest: Wishes become Quests
Date Launch : 1970-01-01 00:00:00
Statut      : submitted


In [5]:
# Filtrer un projet lancé en juillet 2025 et qui était au statut 'live' dans le fichier 202507
mask_july_2025_live = (
    (raw_202507['launched_at'].dt.year == 2025) & 
    (raw_202507['launched_at'].dt.month == 7) & 
    (raw_202507['state'] == 'live')
)

projets_candidats = raw_202507[mask_july_2025_live]

if not projets_candidats.empty:
    # On prend le premier projet trouvé
    projet_202507 = projets_candidats.iloc[0]
    target_id = projet_202507['id']

    # Récupérer ce MÊME projet dans la capture de Juin 2026
    projet_202606 = raw_202606[raw_202606['id'] == target_id]

    print("==================================================")
    print(f"Preuve méthodologique - ID : {target_id}")
    print(f"Nom : {projet_202507['name']}")
    print("==================================================")

    print("\n📍 Statut dans la capture 202507 (Juillet 2025) :")
    print(f"- Launched at : {projet_202507['launched_at']}")
    print(f"- State       : {projet_202507['state']}")

    print("\n📍 Statut dans la capture 202606 (Juin 2026) :")
    if not projet_202606.empty:
        row_2026 = projet_202606.iloc[0]
        print(f"- Launched at : {row_2026['launched_at']}")
        print(f"- State       : {row_2026['state']}")
    else:
        print("Projet non retrouvé dans 202606.")
else:
    print("Aucun projet 'live' lancé en juillet 2025 trouvé dans 202507.")

Preuve méthodologique - ID : 517732119
Nom : Bra Caddy

📍 Statut dans la capture 202507 (Juillet 2025) :
- Launched at : 2025-07-11 01:47:30
- State       : live

📍 Statut dans la capture 202606 (Juin 2026) :
- Launched at : 2025-07-11 01:47:30
- State       : failed


Nous avons donc la preuve qu'il faut se concentrer sur le dataset de **Juin 2026** pour effectuer nos analyses.

In [6]:
raw_202606.info()

<class 'pandas.DataFrame'>
RangeIndex: 272518 entries, 0 to 272517
Data columns (total 42 columns):
 #   Column                              Non-Null Count   Dtype        
---  ------                              --------------   -----        
 0   backers_count                       272518 non-null  int64        
 1   blurb                               272396 non-null  str          
 2   category                            272518 non-null  str          
 3   converted_pledged_amount            252269 non-null  float64      
 4   country                             272518 non-null  str          
 5   country_displayable_name            272518 non-null  str          
 6   created_at                          272518 non-null  datetime64[s]
 7   creator                             272518 non-null  str          
 8   currency                            272518 non-null  str          
 9   currency_symbol                     272518 non-null  str          
 10  currency_trailing_code         

## Dates de lancement

Précédemment, on a vu que certains projets sont vus comme ayant été lancés en 1970. C'est une particularité des timestamp unix : les timestamps démarrent en 1970. Ainsi, si la valeur de la timestamp est à 0 (manquante), alors on verra l'année 1970 au moment de la conversion en *datetime*, ce qui est aberrant car Kickstarter a vu le jour le 28 avril 2009.

In [7]:
# Remplacer toutes les dates aberrantes (avant la création de Kickstarter le 28/04/2009) par NaT
kickstarter_birth = pd.Timestamp('2009-04-28')

for col in cols_dates:
    raw_202606.loc[raw_202606[col] < kickstarter_birth, col] = pd.NaT

# Vérifier en trouvant le projet le plus ancien
valid_launches = raw_202606['launched_at'].dropna()

if not valid_launches.empty:
    idx_min = valid_launches.idxmin()
    projet_plus_ancien = raw_202606.loc[idx_min]

    print("=== PROJET LE PLUS ANCIEN ===")
    print(f"ID          : {projet_plus_ancien['id']}")
    print(f"Nom         : {projet_plus_ancien['name']}")
    print(f"Date Launch : {projet_plus_ancien['launched_at']}")
    print(f"Statut      : {projet_plus_ancien['state']}")
else:
    print("Aucune date de lancement valide trouvée.")

=== PROJET LE PLUS ANCIEN ===
ID          : 2089078683
Nom         : New York Makes a Book!!
Date Launch : 2009-04-28 11:55:41
Statut      : successful


In [8]:
# Quels projets n'ont pas de date de lancement ?
na_launches = raw_202606[raw_202606['launched_at'].isna()]
display(na_launches[['id', 'name', 'state', 'created_at', 'launched_at', 'deadline']])

,id,name,state,created_at,launched_at,deadline
255,848455089,Sous Coffee: Hot Coffee in Every Sip,submitted,2025-08-28 17:03:59,NaT,NaT
275,1649581740,Wrath & Revenge Special Edition Duology,submitted,2025-10-15 16:53:40,NaT,NaT
278,553034339,Deli Duels : Tommy Salami v. Tony Bologna,submitted,2020-06-08 21:22:47,NaT,NaT
279,56785168,"The SuperPoo & Friends, Series Vol. #1",started,2020-02-22 10:39:27,NaT,NaT
298,2071035098,Through Burning Skies: A Gritty Space Opera Ad...,submitted,2025-09-29 20:21:38,NaT,NaT
...,...,...,...,...,...,...
272453,1666958999,Shards of Ember Limited Collector's Edition,submitted,2025-11-30 14:15:20,NaT,NaT
272454,1523334947,The Basketball Firm's - Blueprint Guide to Get...,submitted,2025-11-26 00:28:42,NaT,NaT
272461,309342732,Help Publish the Trooper Series,submitted,2025-11-19 21:26:39,NaT,NaT
272462,1185779878,Viking Children's Book: Erik & Gunnar the Dragon,submitted,2025-11-19 18:39:15,NaT,NaT


In [9]:
# Compter la répartition des statuts pour ces projets
na_launches_states = raw_202606[raw_202606['launched_at'].isna()]['state'].value_counts(dropna=False)

print("=== Répartition des statuts pour les projets sans date de lancement ===")
print(na_launches_states)

=== Répartition des statuts pour les projets sans date de lancement ===
state
submitted     16357
started        3879
suspended        13
successful        1
Name: count, dtype: int64


La plupart des projets sans date de lancement sont :

* `started`: Projets créés mais pas encore soumis à l'équipe de modération Kickstarter.
* `submitted`: Projets créés et soumis à l'équipe de modération Kickstarter pour lancement.
* `suspended`: Projets suspendus par l'équipe de modération Kickstarter.

On voit néanmoins une anomalie de projet notée `successful` alors qu'il n'a pas de date de lancement.

In [10]:
successful_na_launch = raw_202606[
    (raw_202606['launched_at'].isna()) & 
    (raw_202606['state'] == 'successful')
]

display(successful_na_launch[['name', 'id', 'state', 'created_at', 'launched_at']])

,name,id,state,created_at,launched_at
134972,Offline Wikipedia iPhone app,727286,successful,NaT,NaT


Après quelques recherches, j'ai découvert que [ce projet outlier est l'un des tous premiers projets sur Kickstarter](https://www.kickstarter.com/projects/dphiffer/offline-wikipedia-iphone-app/comments). Il a été lancé le 25 avril 2009, quelques jours avant l'ouverture de la plateforme, en guise de test.

Néanmoins, que faire de toutes ces lignes sans date de lancement ?

* Si ces projets n'ont pas été lancés, alors nous ne pouvons pas analyser les facteurs de succès pour trouver son Product Market Fit.
* Par conséquent, ces lignes ne sont pas pertinentes à garder pour notre analyse.

In [11]:
# Supprimer les lignes où launched_at est manquant (NaT / NaN)
raw_202606 = raw_202606.dropna(subset=['launched_at']).copy()

print(f"Nombre de projets restants : {len(raw_202606)}")
print(f"Valeurs manquantes dans launched_at : {raw_202606['launched_at'].isna().sum()}")

Nombre de projets restants : 252268
Valeurs manquantes dans launched_at : 0


## Doublons

Le dataset, avec 252 268 entrées, est volumineux. Voyons si on peut se débarrasser des doublons.

In [12]:
# Nombre de doublons basés sur l'ID
print("Nombre de doublons :", raw_202606['id'].duplicated().sum())

Nombre de doublons : 50834


On va se débarrasser des captures les plus anciennes en se reposant sur la variable `state_changed_at` qui est le dernier changement de statut du projet.

In [13]:
# Convertir la colonne est bien au format datetime
raw_202606['state_changed_at'] = pd.to_datetime(raw_202606['state_changed_at'], errors='coerce')

# Trier par 'id' et par 'state_changed_at' croissant (du plus ancien au plus récent)
raw_202606 = raw_202606.sort_values(by=['id', 'state_changed_at'], ascending=[True, True])

# Supprimer les doublons en ne conservant que la dernière occurrence
raw_202606_unique = raw_202606.drop_duplicates(subset=['id'], keep='last').copy()

print(f"Nombre de projets uniques conservés : {len(raw_202606_unique)}")

Nombre de projets uniques conservés : 201434


## Statistiques basiques

In [14]:
raw_202606_unique.describe(include='all')

,backers_count,blurb,category,converted_pledged_amount,country,country_displayable_name,created_at,creator,currency,currency_symbol,...,spotlight,staff_pick,state,state_changed_at,static_usd_rate,urls,usd_exchange_rate,usd_pledged,usd_type,video
count,201434.000000,201426,201434,2.014340e+05,201434,201434,201431,201434,201434,201434,...,201434,201434,201434,201434,201434.000000,201434,201434.000000,2.014340e+05,201335,133950
unique,NaN,198160,174,NaN,25,25,NaN,200592,15,7,...,2,2,4,NaN,NaN,201434,NaN,NaN,2,133950
top,NaN,"3D Printable STL Files, Terrain for Tabletop, ...","{""id"":34,""name"":""Tabletop Games"",""analytics_na...",NaN,US,the United States,NaN,"{""id"":1712814392,""name"":""Amanda B"",""is_registe...",USD,$,...,True,False,successful,NaN,NaN,"{""web"":{""project"":""https://www.kickstarter.com...",NaN,NaN,domestic,"{""id"":1300562,""status"":""successful"",""hls"":""htt..."
freq,NaN,48,8800,NaN,128654,128654,NaN,8,128663,151714,...,119599,170573,119599,NaN,NaN,1,NaN,NaN,201321,1
mean,132.773474,NaN,NaN,1.661173e+04,NaN,NaN,2019-11-07 05:07:13,NaN,NaN,NaN,...,NaN,NaN,NaN,1970-01-01 00:00:01.580666972,0.984773,NaN,0.984422,1.661177e+04,NaN,NaN
min,0.000000,NaN,NaN,0.000000e+00,NaN,NaN,2009-04-29 09:32:34,NaN,NaN,NaN,...,NaN,NaN,NaN,1970-01-01 00:00:01.242468025,0.006209,NaN,0.006186,0.000000e+00,NaN,NaN
25%,5.000000,NaN,NaN,1.740000e+02,NaN,NaN,2015-11-24 18:55:25,NaN,NaN,NaN,...,NaN,NaN,NaN,1970-01-01 00:00:01.455942594,1.000000,NaN,1.000000,1.747351e+02,NaN,NaN
50%,28.000000,NaN,NaN,1.834000e+03,NaN,NaN,2019-09-03 21:02:48,NaN,NaN,NaN,...,NaN,NaN,NaN,1970-01-01 00:00:01.574495970,1.000000,NaN,1.000000,1.835000e+03,NaN,NaN
75%,90.000000,NaN,NaN,7.407000e+03,NaN,NaN,2024-03-14 15:50:24,NaN,NaN,NaN,...,NaN,NaN,NaN,1970-01-01 00:00:01.718731997,1.000000,NaN,1.000000,7.403000e+03,NaN,NaN
max,105857.000000,NaN,NaN,4.676226e+07,NaN,NaN,2026-06-10 19:29:39,NaN,NaN,NaN,...,NaN,NaN,NaN,1970-01-01 00:00:01.781153820,1.716408,NaN,1.716408,4.676226e+07,NaN,NaN


## Top 10 des projets les plus performants

In [15]:
# Enlever notation scientifique

pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [16]:
# On ne veut que les succès.
successful_sample = raw_202606_unique[raw_202606_unique['state'] == 'successful']

# Top 10 par pourcentage de financement (%)
top10_percent = successful_sample.sort_values(by='percent_funded', ascending=False).head(10)

print("Top 10 : Pourcentage de financement (%)")
col_usd = 'usd_pledged' if 'usd_pledged' in raw_202606_unique.columns else 'converted_pledged_amount'
display(top10_percent[['id', 'name', 'percent_funded', col_usd]])

print("-----------------------------------")

# Top 10 par somme récoltée en USD
top10_usd = successful_sample.sort_values(by=col_usd, ascending=False).head(10)

print("Top 10 : Somme récoltée (USD)")
display(top10_usd[['id', 'name', col_usd, 'percent_funded']])

Top 10 : Pourcentage de financement (%)


,id,name,percent_funded,usd_pledged
235527,149159163,AdventureQuest Worlds: Infinity,214940400.00,2149404.00
116005,649845747,Laurent Garnier: Off the Record,15532039.00,195650.15
133748,256471901,Voyage - New Album Campaign by The Longest Johns,13806700.00,175331.19
245128,1947298033,Re-covering with Friends,6876410.00,68764.10
49459,1824638248,The Swapper3D: NO MORE PURGE BLOCKS!,5500829.00,42730.16
117186,712425525,LET ME HOLD A DOLLA! | LARUSSELL GOES MAJORLY ...,3942300.00,39423.00
259912,246326560,Tim Rose Visual Album: <3,2868818.00,28688.18
139338,521903377,CLOCKWRIGHT: Large-Scale Analogue Time Machines,2758823.00,27588.23
141461,550443638,"Penny Arcade's Podcast, ""Downloadable Content""...",2303606.70,230360.67
133179,942323643,"Multi-Purpose, All-Occasion Greeting Cards",1257515.00,25150.30


-----------------------------------
Top 10 : Somme récoltée (USD)


,id,name,usd_pledged,percent_funded
201096,1812062326,eufyMake E1: the First Personal 3D-Texture UV ...,46762258.00,9352.45
194077,339473088,The Official Cyberpunk Trading Card Game,28353088.00,28353.09
38683,411573180,Snapmaker U1 Color 3D Printer: 5X More Speed. ...,20614548.00,20614.55
58804,511374686,XGIMI TITAN Noir Series: A Dual Iris 4K RGB La...,18842264.00,18842.26
113660,1850458289,AWOL Vision Aetherion: Pixel-Clarity RGB Laser...,18649456.00,3729.89
252436,870254269,NestWorks C500 - Next-Gen Smartest CNC with In...,12946959.73,25893.69
225492,235978012,"Makera Z1 Desktop CNC: Entry-level CNC, Pro-le...",12432266.59,12432.56
213173,1198848775,EcoFlow DELTA Pro: The Portable Home Battery,12179651.05,12179.65
209441,597041739,Critical Role: The Legend of Vox Machina Anima...,11385449.05,1518.06
241974,1431038175,The Smith Blade (21-in-1 Titanium Multi-Tool),11141193.56,6145.67


## Enrichissement

### Variables temporelles
En vue d'analyser les axes suivants :

* La durée des campagnes a-t-elle un impact sur leur succès ?
* La saisonnalité des lancements, certains mois génèrent-ils de meilleurs taux de financement ?

In [17]:
# Ajout d'une colonne pour montrer la durée en jours
raw_202606_unique['duration_days'] = (raw_202606_unique['deadline'] - raw_202606_unique['launched_at']).dt.days

# Ajout d'une colonne pour montrer la durée de préparation de la campagne
raw_202606_unique['prep_days'] = (raw_202606_unique['launched_at'] - raw_202606_unique['created_at']).dt.days
raw_202606_unique['prep_days'] = raw_202606_unique['prep_days'].fillna(0).astype('int64')

# Ajout d'une colonne pour voir le mois de lancement
raw_202606_unique['launch_month'] = raw_202606_unique['launched_at'].dt.month_name()

# Ajout d'une colonne pour voir l'année de lancement
raw_202606_unique['launch_year'] = raw_202606_unique['launched_at'].dt.year

display(raw_202606_unique[['id', 'name', 'duration_days', 'prep_days', 'launch_month', 'launch_year']].head(5))

print(raw_202606_unique['launch_year'].value_counts().sort_index())
print("-" * 50)
print(raw_202606_unique['duration_days'].describe())

,id,name,duration_days,prep_days,launch_month,launch_year
89966,13583,American Dream # 1 | 31 Pages of Full Color He...,60,520,September,2024
6979,18520,Grandma's are Life,30,0,October,2016
133802,19685,Pumpkin & Friends: Soft & Silly Plush Pals!,45,35,May,2024
155929,19795,Lost Mine - Dice Tower - 3D Printable STL Files,7,0,November,2025
41214,21109,Meta,28,0,April,2015


launch_year
2009      151
2010      872
2011     2467
2012     4640
2013     5376
2014    15724
2015    20406
2016    14918
2017    14233
2018    12354
2019    11455
2020     9656
2021     9519
2022    10355
2023    11447
2024    20419
2025    24549
2026    12893
Name: count, dtype: int64
--------------------------------------------------
count   201434.00
mean        33.09
std         12.89
min          1.00
25%         29.00
50%         30.00
75%         36.00
max        120.00
Name: duration_days, dtype: float64


On remarque un outlier au niveau de la durée. 120 jours n'a jamais été une campagne configurable ; en effet, depuis le 17 juin 2011, la durée maximale des campagnes est de 60 jours. Auparavant, elle était de 90 jours.

In [18]:
anomalies_duree = raw_202606_unique[raw_202606_unique['duration_days'] == 120]

display(anomalies_duree.head().sort_index())

,backers_count,blurb,category,converted_pledged_amount,country,country_displayable_name,created_at,creator,currency,currency_symbol,...,static_usd_rate,urls,usd_exchange_rate,usd_pledged,usd_type,video,duration_days,prep_days,launch_month,launch_year
73790,14,When gaming meets fashion to create durable ha...,"{""id"":263,""name"":""Apparel"",""analytics_name"":""A...",2797.00,US,the United States,2020-10-16 23:12:09,"{""id"":1719709121,""name"":""Luca Designs"",""is_reg...",USD,$,...,1.00,"{""web"":{""project"":""https://www.kickstarter.com...",1.00,2797.00,domestic,NaN,120,9,October,2020


En vérifiant sur Kickstarter, en effet cette campagne a duré 120 jours mais c'est bien la seule, je n'ai pas trouvé d'explication. Je choisis de la garder car elle a été réussie, néanmoins j'exclurai sa durée du champ d'analyse car il s'agit d'un outlier.

### Variables financières
En vue d'analyser les axes suivants :

* Comparer la distribution des objectifs fixés entre les projets réussis et échouées. Les projets avec des objectifs ambitieux ont-ils un taux de conversion supérieur ?
* Taux de surfinancement : étudier la distribution de `percent_funded` sur l'ensemble des projets successful pour voir si la surperformance est marginale ou courante.

Nous sommes en France et notre monnaie est l'euro. La première étape est de convertir les variables financières en EUR pour une meilleure lisibilité.

In [19]:
import numpy as np

# Calcul d'une variable 'goal_usd' car le goal est en devise d'origine
# Taux implicite USD / devise d'origine
usd_fx = raw_202606_unique['usd_pledged'] / raw_202606_unique['pledged']

# Si pledged == 0, on utilise fx_rate en fallback
raw_202606_unique['goal_usd'] = np.where(
    raw_202606_unique['pledged'] > 0,
    raw_202606_unique['goal'] * usd_fx,
    raw_202606_unique['goal'] * raw_202606_unique['fx_rate']
)

# Calcul de 'goal_eur' et 'pledged_eur'
# Taux USD vers EUR au 21/07/2026
usd_to_eur_rate = 0.88

raw_202606_unique['goal_eur'] = np.where(
    raw_202606_unique['currency'] == 'EUR',
    raw_202606_unique['goal'],
    raw_202606_unique['goal_usd'] * usd_to_eur_rate
)

raw_202606_unique['pledged_eur'] = np.where(
    raw_202606_unique['currency'] == 'EUR',
    raw_202606_unique['pledged'],
    raw_202606_unique['usd_pledged'] * usd_to_eur_rate
)

display(raw_202606_unique[['id', 'name', 'goal_eur', 'goal_usd', 'goal', 'currency']].head(5))
print("On voit désormais les montants en euro.")

,id,name,goal_eur,goal_usd,goal,currency
89966,13583,American Dream # 1 | 31 Pages of Full Color He...,440.00,500.00,500.00,USD
6979,18520,Grandma's are Life,13200.00,15000.00,15000.00,USD
133802,19685,Pumpkin & Friends: Soft & Silly Plush Pals!,2200.00,2500.00,2500.00,USD
155929,19795,Lost Mine - Dice Tower - 3D Printable STL Files,88.00,100.00,100.00,USD
41214,21109,Meta,196.88,223.73,150.00,GBP


On voit désormais les montants en euro.


### Dynamique de communauté
* Identifier le panier moyen `pledged_eur` / `backers_count` par projet
* Définir profil type de backer en fonction du panier moyen

In [20]:

raw_202606_unique['avg_basket_eur'] = np.where(
    raw_202606_unique['backers_count'] > 0,
    raw_202606_unique['pledged_eur'] / raw_202606_unique['backers_count'],
    0
)

valid_pledges = raw_202606_unique[raw_202606_unique['avg_basket_eur'] > 0]['avg_basket_eur']

quartiles = valid_pledges.quantile([0, 0.25, 0.50, 0.75, 1.0])
print(f"Min (0%)    : {quartiles[0.00]:.2f} €")
print(f"Q1  (25%)   : {quartiles[0.25]:.2f} €  -> Micro-mécénat")
print(f"Q2  (50%)   : {quartiles[0.50]:.2f} €  -> Client standard")
print(f"Q3  (75%)   : {quartiles[0.75]:.2f} €  -> Acheteur requis")
print(f"Max (100%)  : {quartiles[1.00]:.2f} €  -> Grand mécène")

Min (0%)    : 0.40 €
Q1  (25%)   : 25.99 €  -> Micro-mécénat
Q2  (50%)   : 49.75 €  -> Client standard
Q3  (75%)   : 89.15 €  -> Acheteur requis
Max (100%)  : 9651.02 €  -> Grand mécène


In [21]:
# Définition des 4 tranches automatiques + 1 tranche dédiée aux 0€
raw_202606_unique['backer_profile_qcut'] = pd.qcut(
    valid_pledges,
    q=4,
    labels=['Micro-mécénat (< 25,99€)', 'Client classique (25,99 - 49,75€)', 'Acheteur requis (49,75 - 89,15€)', 'Grand mécène (89,15€ <)']
)

raw_202606_unique['backer_profile_qcut'] = raw_202606_unique['backer_profile_qcut'].astype(str)
raw_202606_unique.loc[raw_202606_unique['avg_basket_eur'] == 0, 'backer_profile_qcut'] = 'Aucun soutien (0€)'

print(raw_202606_unique['backer_profile_qcut'].value_counts())

backer_profile_qcut
Micro-mécénat (< 25,99€)             47333
Grand mécène (89,15€ <)              47333
Client classique (25,99 - 49,75€)    47333
Acheteur requis (49,75 - 89,15€)     47332
Aucun soutien (0€)                   12103
Name: count, dtype: int64


### Analyse par Catégorie et Sous-catégorie
La variable `category` est un morceau de JSON dont il faut extraire la catégorie et la sous-catégorie.

* Taux de succès par catégorie
* Montant moyen récolté par catégorie
* `percent_funded` par catégorie

In [22]:
import json

def extract_categories(val):
    if pd.isna(val):
        return pd.Series([None, None])
    
    if isinstance(val, str):
        try:
            val = json.loads(val)
        except:
            return pd.Series([None, None])
            
    if isinstance(val, dict):
        main_cat = val.get('parent_name') or val.get('analytics_name')
        
        sub_cat = val.get('analytics_name')
        
        return pd.Series([main_cat, sub_cat])
        
    return pd.Series([None, None])

raw_202606_unique[['main_category', 'sub_category']] = raw_202606_unique['category'].apply(extract_categories)

In [23]:
display(raw_202606_unique[['id', 'name', 'main_category', 'sub_category']].head(5))

,id,name,main_category,sub_category
89966,13583,American Dream # 1 | 31 Pages of Full Color He...,Comics,Comic Books
6979,18520,Grandma's are Life,Music,World Music
133802,19685,Pumpkin & Friends: Soft & Silly Plush Pals!,Design,Toys
155929,19795,Lost Mine - Dice Tower - 3D Printable STL Files,Games,Tabletop Games
41214,21109,Meta,Art,Performance Art


In [24]:
raw_202606_unique['cat_mean_success_rate'] = raw_202606_unique.groupby('main_category')['state'].transform(
    lambda x: (x == 'successful').mean() * 100
)

raw_202606_unique['cat_median_pledged_eur'] = raw_202606_unique.groupby('main_category')['pledged_eur'].transform('median')

raw_202606_unique['outperformed_cat_median_eur'] = raw_202606_unique['pledged_eur'] > raw_202606_unique['cat_median_pledged_eur']

display(raw_202606_unique[['id', 'name', 'main_category', 'sub_category', 'cat_median_pledged_eur', 'outperformed_cat_median_eur', 'cat_mean_success_rate']].head(5))

,id,name,main_category,sub_category,cat_median_pledged_eur,outperformed_cat_median_eur,cat_mean_success_rate
89966,13583,American Dream # 1 | 31 Pages of Full Color He...,Comics,Comic Books,2904.44,False,77.03
6979,18520,Grandma's are Life,Music,World Music,2336.40,False,70.19
133802,19685,Pumpkin & Friends: Soft & Silly Plush Pals!,Design,Toys,3304.93,True,63.71
155929,19795,Lost Mine - Dice Tower - 3D Printable STL Files,Games,Tabletop Games,1924.04,False,70.45
41214,21109,Meta,Art,Performance Art,1436.70,False,71.40


### Dimension géographique
* Analyse de la provenance des projets
* Taux de succès selon la géographie

Pour la vue macro, on peut déjà chercher à enrichir en découpant par région du monde.

In [25]:
def map_country_to_region(country_code):
    if pd.isna(country_code):
        return 'Non renseigné'
    
    code = str(country_code).upper().strip()
    
    if code in ['CA', 'MX', 'US']:
        return 'Amérique du Nord'
    elif code in ['GB', 'FR', 'DE', 'IT', 'ES', 'NL', 'BE', 'CH', 'AT', 'SE', 'DK', 'NO', 'FI', 'IE', 'PL', 'LU']:
        return 'Europe'
    elif code in ['AU', 'NZ']:
        return 'Océanie'
    elif code in ['HK', 'SG', 'JP']:
        return 'Asie'
    else:
        return 'Reste du monde'

raw_202606_unique['region'] = raw_202606_unique['country'].apply(map_country_to_region)

raw_202606_unique['region_success_rate'] = raw_202606_unique.groupby('region')['state'].transform(
    lambda x: (x == 'successful').mean() * 100
)

print("=== RÉPARTITION DES PROJETS PAR RÉGION ===")
print(raw_202606_unique['region'].value_counts())

=== RÉPARTITION DES PROJETS PAR RÉGION ===
region
Amérique du Nord    141723
Europe               47392
Asie                  5854
Océanie               5671
Reste du monde         794
Name: count, dtype: int64


In [26]:
country_mapping = {
    'US': 'États-Unis',
    'GB': 'Royaume-Uni',
    'CA': 'Canada',
    'AU': 'Australie',
    'NZ': 'Nouvelle-Zélande',
    'NL': 'Pays-Bas',
    'DK': 'Danemark',
    'IE': 'Irlande',
    'NO': 'Norvège',
    'SE': 'Suède',
    'DE': 'Allemagne',
    'FR': 'France',
    'ES': 'Espagne',
    'IT': 'Italie',
    'AT': 'Autriche',
    'BE': 'Belgique',
    'CH': 'Suisse',
    'LU': 'Luxembourg',
    'HK': 'Hong Kong',
    'SG': 'Singapour',
    'MX': 'Mexique',
    'JP': 'Japon',
    'GR': 'Grèce',
    'PL': 'Pologne',
    'SI': 'Slovénie',
    'N,0"': 'Non défini' 
}

raw_202606_unique['country'] = raw_202606_unique['country'].replace(country_mapping)

print(raw_202606_unique['country'].value_counts())

country
États-Unis          128654
Royaume-Uni          23304
Canada                9533
Australie             4837
Allemagne             4700
France                3739
Italie                3613
Mexique               3536
Espagne               3457
Hong Kong             3404
Pays-Bas              1733
Japon                 1536
Suède                 1491
Pologne               1068
Singapour              914
Danemark               864
Nouvelle-Zélande       834
Suisse                 804
Irlande                762
Belgique               759
Grèce                  708
Autriche               568
Norvège                453
Slovénie                86
Luxembourg              77
Name: count, dtype: int64


### Théorie du Signal
La théorie du signal (Spence, 1973) stipule que dans une situation d'asymétrie d'information (comme Kickstarter, où le backer ne sait pas si le porteur de projet est sérieux), le créateur doit envoyer des signaux de qualité pour rassurer les contributeurs.

* Présence ou non d'une vidéo explicative
* Longueur du blurb et du nom

In [27]:
def check_has_video(val):
    if pd.isna(val) or val is None or val == '' or val == False:
        return False
    return True

raw_202606_unique['has_video'] = raw_202606_unique['video'].apply(check_has_video)

raw_202606_unique['name_len_char'] = raw_202606_unique['name'].fillna('').astype(str).str.len()
raw_202606_unique['name_len_words'] = raw_202606_unique['name'].fillna('').astype(str).str.split().str.len()

raw_202606_unique['blurb_len_char'] = raw_202606_unique['blurb'].fillna('').astype(str).str.len()
raw_202606_unique['blurb_len_words'] = raw_202606_unique['blurb'].fillna('').astype(str).str.split().str.len()

raw_202606_unique['blurb_density'] = np.where(
    raw_202606_unique['blurb_len_words'] > 0,
    raw_202606_unique['blurb_len_char'] / raw_202606_unique['blurb_len_words'],
    0
)

In [28]:
cols_signal = [
    'name', 'has_video', 
    'name_len_words', 'blurb_len_char', 'blurb_len_words', 
    'blurb_density'
]
display(raw_202606_unique[cols_signal].head(10))

,name,has_video,name_len_words,blurb_len_char,blurb_len_words,blurb_density
89966,American Dream # 1 | 31 Pages of Full Color He...,True,12,97,17,5.71
6979,Grandma's are Life,False,3,135,24,5.62
133802,Pumpkin & Friends: Soft & Silly Plush Pals!,False,8,58,10,5.80
155929,Lost Mine - Dice Tower - 3D Printable STL Files,False,10,79,11,7.18
41214,Meta,False,1,129,24,5.38
151126,The POWER of Good Decisions,False,5,34,6,5.67
34992,Puss N' Books: A relaxing cat cafe and bookstore.,False,9,133,24,5.54
69974,TASTE MAKERS BY TRISH P,True,5,134,23,5.83
243330,The Meat Candy Experience,False,4,86,13,6.62
199980,Orage: Interactive Lightning Experience at Bur...,True,8,126,18,7.00


## Tri des variables

In [29]:
cols_to_keep = [
    'id', 'name', 'blurb', 'state', 
    'created_at', 'launched_at', 'deadline', 
    'duration_days', 'prep_days', 'launch_month', 'launch_year',
    'goal_eur', 'pledged_eur', 'percent_funded', 
    'backers_count', 'avg_basket_eur', 'backer_profile_qcut', 'prelaunch_activated',
    'main_category', 'sub_category', 'cat_median_pledged_eur',
    'country', 'region',
    'has_video', 'staff_pick', 'spotlight',
    'name_len_words', 'name_len_char', 'blurb_len_char', 'blurb_len_words', 'blurb_density'
]


raw_202606_unique = raw_202606_unique[cols_to_keep].copy()

print(f"Nouveau nombre de colonnes : {raw_202606_unique.shape[1]}")
raw_202606_unique.info()

Nouveau nombre de colonnes : 31
<class 'pandas.DataFrame'>
Index: 201434 entries, 89966 to 108618
Data columns (total 31 columns):
 #   Column                  Non-Null Count   Dtype        
---  ------                  --------------   -----        
 0   id                      201434 non-null  int64        
 1   name                    201434 non-null  str          
 2   blurb                   201426 non-null  str          
 3   state                   201434 non-null  str          
 4   created_at              201431 non-null  datetime64[s]
 5   launched_at             201434 non-null  datetime64[s]
 6   deadline                201434 non-null  datetime64[s]
 7   duration_days           201434 non-null  int64        
 8   prep_days               201434 non-null  int64        
 9   launch_month            201434 non-null  str          
 10  launch_year             201434 non-null  int32        
 11  goal_eur                201434 non-null  float64      
 12  pledged_eur             

## Deuxième passe sur les valeurs manquantes

In [30]:
missing_summary = pd.DataFrame({
    'Manquants (N)': raw_202606_unique.isnull().sum(),
    'Pourcentage (%)': (raw_202606_unique.isnull().sum() / len(raw_202606_unique) * 100).round(2)
})

missing_summary = missing_summary[missing_summary['Manquants (N)'] > 0].sort_values(by='Manquants (N)', ascending=False)

print("Valeurs manquantes")
display(missing_summary)

Valeurs manquantes


,Manquants (N),Pourcentage (%)
blurb,8,0.00
created_at,3,0.00


Les valeurs manquantes pour les variables temporelles sont des projets qui n'ont jamais été lancés. Pour analyser les facteurs de succès d'un projet en vue de valider un Product Market Fit, les projets doivent à minima avoir été lancés : il faut donc supprimer ces lignes.

In [31]:
raw_202606_unique.dropna(subset=['created_at'], inplace=True)

missing_summary = pd.DataFrame({
    'Manquants (N)': raw_202606_unique.isnull().sum(),
    'Pourcentage (%)': (raw_202606_unique.isnull().sum() / len(raw_202606_unique) * 100).round(2)
})

missing_summary = missing_summary[missing_summary['Manquants (N)'] > 0].sort_values(by='Manquants (N)', ascending=False)

print("Valeurs manquantes")
display(missing_summary)

Valeurs manquantes


,Manquants (N),Pourcentage (%)
blurb,8,0.00


Les valeurs manquantes qui restent sont des projets pour lesquels il n'y a pas eu de blurb. On va tout simplement remplir par "Sans blurb". Le fait de ne pas renseigner de blurb est sans doute un facteur qui influence le succès d'une campagne.

In [32]:
missing_blurb_mask = raw_202606_unique['blurb'].isna()

raw_202606_unique['blurb'] = raw_202606_unique['blurb'].fillna('Sans blurb')

raw_202606_unique.loc[missing_blurb_mask, ['blurb_len_char', 'blurb_density']] = 0

raw_202606_unique['blurb_density'] = np.where(
    raw_202606_unique['blurb_len_words'] > 0,
    raw_202606_unique['blurb_len_char'] / raw_202606_unique['blurb_len_words'],
    0
)


In [33]:
check_blurb = raw_202606_unique[raw_202606_unique['blurb'] == 'Sans blurb']
display(check_blurb[['name', 'blurb', 'blurb_len_char', 'blurb_len_words', 'blurb_density']])

missing_summary = pd.DataFrame({
    'Manquants (N)': raw_202606_unique.isnull().sum(),
    'Pourcentage (%)': (raw_202606_unique.isnull().sum() / len(raw_202606_unique) * 100).round(2)
})

missing_summary = missing_summary[missing_summary['Manquants (N)'] > 0].sort_values(by='Manquants (N)', ascending=False)

print("Valeurs manquantes")
display(missing_summary)
print("Il n'y a plus de valeur manquante.")

,name,blurb,blurb_len_char,blurb_len_words,blurb_density
106216,Vending Machine (Canceled),Sans blurb,0,0,0.00
140535,Star Wars Bluetooth Speakers (Canceled),Sans blurb,0,0,0.00
42625,Ready to wear,Sans blurb,0,0,0.00
226759,Cancelled,Sans blurb,0,0,0.00
167923,Cammy The Colorful Chameleon,Sans blurb,0,0,0.00
159642,N/A (Canceled),Sans blurb,0,0,0.00
265537,Omaxs VentoGo X1:Your 4-in-1 Car Detailing & E...,Sans blurb,0,0,0.00
159635,N/A (Canceled),Sans blurb,0,0,0.00


Valeurs manquantes


,Manquants (N),Pourcentage (%)


Il n'y a plus de valeur manquante.


### Le cas des titres de projet et des blurb avec "Cancelled"

Le précédent aperçu nous montre que beaucoup de projets ont été annulés. Leurs noms et leurs blurb sont parfois marqués "Cancelled" ou N/A Canceled. Pour traiter proprement ces cas de figure, il faut retirer les mentions d'annulation et remplacer les titres N/A par "Sans Titre" pour analyser correctement la longueur du nom.



In [34]:
import re 
raw_202606_unique['name'] = (
    raw_202606_unique['name']
    .str.replace(r'\s*\((canceled|cancelled)\)$', '', regex=True, flags=re.IGNORECASE)
    .str.strip() 
)
invalid_names = ['', 'N/A', 'n/a', 'Cancelled', 'Canceled', 'cancelled', 'canceled', 'None', 'nan']

raw_202606_unique['name'] = raw_202606_unique['name'].replace(invalid_names, 'Sans titre')

masque_canceled = raw_202606_unique['name'].str.contains('canceled|cancelled', case=False, na=False)

nb_restants = masque_canceled.sum()
print(f"Nombre de projets contenant encore 'canceled' ou 'cancelled' : {nb_restants}")

raw_202606_unique[masque_canceled][['id', 'name', 'state']]

Nombre de projets contenant encore 'canceled' ou 'cancelled' : 22


,id,name,state
204158,39200802,"Sorry, but this project has been cancelled",canceled
204756,123802505,"Dice Set - Foulebo Creation (canceled, error c...",canceled
137399,134217336,Project CANCELED,canceled
226639,491796684,Page cancelled due to setup error,canceled
173924,761045131,This project has been cancelled,failed
58763,805154675,Cancelled.,canceled
23205,1013378549,Divine Retribution - Director's Cut - Cancelled,canceled
22528,1088675178,#CANCELLED,failed
149433,1213751549,This project has been cancelled by its creators,canceled
84267,1222381841,Bring Back QUEER KID STUFF! (version 1 CANCELLED),canceled


In [35]:
def clean_project_name(name):
    if not isinstance(name, str) or not name.strip():
        return "Sans Titre"
   
    if re.search(r'^(page\s+)?cance[ll]+ed\s+due\s+to', name.strip(), re.IGNORECASE):
        return "Sans Titre"
    
    cleaned = re.sub(r'\s*\(\s*cance[ll]+ed[^\)]*\)', '', name, flags=re.IGNORECASE)
    
    cleaned = re.sub(r'[\s\-_–\(\[]*(project\s+)?cance[ll]+ed[\s\)\.\!\]]*$', '', cleaned, flags=re.IGNORECASE)
    
    cleaned = re.sub(r'^\s*cance[ll]+ed[\s\-_–:]*', '', cleaned, flags=re.IGNORECASE)
    
    cleaned = cleaned.strip()
    
    if not cleaned or cleaned.lower() in ['n/a', 'none', 'nan', 'cancelled', 'canceled']:
        return "Sans Titre"
        
    return cleaned

raw_202606_unique['name'] = raw_202606_unique['name'].apply(clean_project_name)

nb_restants = masque_canceled.sum()
print(f"Nombre de projets contenant encore 'canceled' ou 'cancelled' : {nb_restants}")

raw_202606_unique[masque_canceled][['id', 'name', 'state']]

Nombre de projets contenant encore 'canceled' ou 'cancelled' : 22


,id,name,state
204158,39200802,"Sorry, but this project has been",canceled
204756,123802505,Dice Set - Foulebo Creation,canceled
137399,134217336,Sans Titre,canceled
226639,491796684,Sans Titre,canceled
173924,761045131,This project has been,failed
58763,805154675,Sans Titre,canceled
23205,1013378549,Divine Retribution - Director's Cut,canceled
22528,1088675178,#,failed
149433,1213751549,This project has been cancelled by its creators,canceled
84267,1222381841,Bring Back QUEER KID STUFF! (version 1,canceled


Pour le titre qui reste, on le garde car il s'agit d'un titre créatif.

In [36]:
raw_202606_unique['blurb'] = (
    raw_202606_unique['blurb']
    .str.replace(r'\s*\((canceled|cancelled)\)$', '', regex=True, flags=re.IGNORECASE)
    .str.strip() 
)

invalid_blurbs = ['', 'N/A', 'n/a', 'Cancelled', 'Canceled', 'cancelled', 'canceled', 'None', 'nan']

raw_202606_unique['blurb'] = raw_202606_unique['blurb'].replace(invalid_names, 'Sans blurb')

masque_canceled_blurb = raw_202606_unique['blurb'].str.contains('canceled|cancelled', case=False, na=False)

nb_restants = masque_canceled_blurb.sum()
print(f"Nombre de blurbs contenant encore 'canceled' ou 'cancelled' : {nb_restants}")

raw_202606_unique[masque_canceled_blurb][['id', 'blurb', 'state']]

Nombre de blurbs contenant encore 'canceled' ou 'cancelled' : 30


,id,blurb,state
35661,391083153,cancelled until later date,canceled
143371,425994109,"Help Cancel Cancel Culture, by sharing the sto...",successful
214582,456269004,We started then canceled this kickstarter 8 da...,successful
37424,733696741,Did you ever have a favorite show get canceled...,successful
58763,805154675,Cancelled.,canceled
223347,914427934,Great 2D action game! The return of the cancel...,successful
95282,957335195,Browncoats rise again with a new album of musi...,successful
211994,981837649,A troubled teen discovers a witch's bottle con...,canceled
133071,1036647445,Sonic Underground was a TV show created by DIC...,failed
129228,1043629990,The Pyramids will tour Europe Nov. 25 - Dec. 1...,successful


In [37]:
def clean_project_blurb(blurb):
    if not isinstance(blurb, str) or not blurb.strip():
        return "Sans blurb"
    
    txt = blurb.strip()
   
    generic_patterns = [
        r'^\*?cance[ll]+ed\.?$',
        r'^\*?(this\s+)?(project|campaign|kickstarter|page)\s+(was\s+)?cance[ll]+ed.*$',
        r'^cance[ll]+ed\s+(project|campaign|page|funding).*$',
        r'^cance[ll]+ed\s+(until|due\s+to|for|by).*$',
        r'^kickstarter\s+campaign\s+cance[ll]+ed.*$'
    ]
    
    for pattern in generic_patterns:
        if re.match(pattern, txt, re.IGNORECASE):
            return "Sans blurb"
 
    cleaned = re.sub(r'[\s\-_–\(\[]*(project\s+|campaign\s+)?cance[ll]+ed[\s\)\.\!\]]*$', '', txt, flags=re.IGNORECASE).strip()
    
    invalid_blurbs = ['', 'n/a', 'none', 'nan', '.', '...', 'cancelled', 'canceled']
    if not cleaned or cleaned.lower() in invalid_blurbs:
        return "Sans blurb"
        
    return cleaned

raw_202606_unique['blurb'] = raw_202606_unique['blurb'].apply(clean_project_blurb)

nb_restants = masque_canceled_blurb.sum()
print(f"Nombre de blurbs contenant encore 'canceled' ou 'cancelled' : {nb_restants}")

raw_202606_unique[masque_canceled_blurb][['id', 'blurb', 'state']]

Nombre de blurbs contenant encore 'canceled' ou 'cancelled' : 30


,id,blurb,state
35661,391083153,Sans blurb,canceled
143371,425994109,"Help Cancel Cancel Culture, by sharing the sto...",successful
214582,456269004,We started then canceled this kickstarter 8 da...,successful
37424,733696741,Did you ever have a favorite show get canceled...,successful
58763,805154675,Sans blurb,canceled
223347,914427934,Great 2D action game! The return of the cancel...,successful
95282,957335195,Browncoats rise again with a new album of musi...,successful
211994,981837649,A troubled teen discovers a witch's bottle con...,canceled
133071,1036647445,Sonic Underground was a TV show created by DIC...,failed
129228,1043629990,The Pyramids will tour Europe Nov. 25 - Dec. 1...,successful


## Arrondir les floats

In [38]:
float_cols = raw_202606_unique.select_dtypes(include=['float64', 'float32']).columns
raw_202606_unique[float_cols] = raw_202606_unique[float_cols].round(2)

## Variable is_success

In [39]:
raw_202606_unique['is_success'] = (raw_202606_unique['state'] == 'successful').astype(bool)

## Export du dataset enrichi et nettoyé

In [40]:
clean_kickstarter = raw_202606_unique

In [41]:
clean_kickstarter.info()

<class 'pandas.DataFrame'>
Index: 201431 entries, 89966 to 108618
Data columns (total 32 columns):
 #   Column                  Non-Null Count   Dtype        
---  ------                  --------------   -----        
 0   id                      201431 non-null  int64        
 1   name                    201431 non-null  str          
 2   blurb                   201431 non-null  str          
 3   state                   201431 non-null  str          
 4   created_at              201431 non-null  datetime64[s]
 5   launched_at             201431 non-null  datetime64[s]
 6   deadline                201431 non-null  datetime64[s]
 7   duration_days           201431 non-null  int64        
 8   prep_days               201431 non-null  int64        
 9   launch_month            201431 non-null  str          
 10  launch_year             201431 non-null  int32        
 11  goal_eur                201431 non-null  float64      
 12  pledged_eur             201431 non-null  float64      
 

In [42]:
clean_kickstarter['launch_year'] = clean_kickstarter['launch_year'].astype('int32')
clean_kickstarter.info()

<class 'pandas.DataFrame'>
Index: 201431 entries, 89966 to 108618
Data columns (total 32 columns):
 #   Column                  Non-Null Count   Dtype        
---  ------                  --------------   -----        
 0   id                      201431 non-null  int64        
 1   name                    201431 non-null  str          
 2   blurb                   201431 non-null  str          
 3   state                   201431 non-null  str          
 4   created_at              201431 non-null  datetime64[s]
 5   launched_at             201431 non-null  datetime64[s]
 6   deadline                201431 non-null  datetime64[s]
 7   duration_days           201431 non-null  int64        
 8   prep_days               201431 non-null  int64        
 9   launch_month            201431 non-null  str          
 10  launch_year             201431 non-null  int32        
 11  goal_eur                201431 non-null  float64      
 12  pledged_eur             201431 non-null  float64      
 

In [43]:
clean_kickstarter.describe()

,id,created_at,launched_at,deadline,duration_days,prep_days,launch_year,goal_eur,pledged_eur,percent_funded,backers_count,avg_basket_eur,cat_median_pledged_eur,name_len_words,name_len_char,blurb_len_char,blurb_len_words,blurb_density
count,201431.00,201431,201431,201431,201431.00,201431.00,201431.00,201431.00,201431.00,201431.00,201431.00,201431.00,201431.00,201431.00,201431.00,201431.00,201431.00,201431.00
mean,1075339226.42,2019-11-07 05:07:13,2020-01-01 23:59:40,2020-02-04 07:10:38,33.09,55.30,2019.52,32441.83,14631.64,1888.35,132.77,81.64,1788.63,5.86,35.99,104.45,17.26,6.17
min,13583.00,2009-04-29 09:32:34,2009-04-29 11:52:03,2009-06-06 05:00:00,1.00,0.00,2009.00,0.01,0.00,0.00,0.00,0.00,90.64,1.00,1.00,0.00,0.00,0.00
25%,539537809.00,2015-11-24 18:55:25,2016-01-20 00:36:06,2016-02-21 19:24:23,29.00,3.00,2016.00,1113.09,154.00,4.18,5.00,21.58,1359.80,4.00,23.00,85.00,14.00,5.55
50%,1075640924.00,2019-09-03 21:02:48,2019-10-21 15:25:53,2019-11-25 01:48:39,30.00,14.00,2019.00,3951.50,1616.56,103.18,28.00,46.36,1924.04,6.00,36.00,117.00,18.00,6.05
75%,1611566993.50,2024-03-14 15:50:24,2024-05-17 17:00:24,2024-06-19 22:13:30,36.00,44.00,2024.00,10000.00,6521.15,143.88,90.00,85.99,2336.40,8.00,50.00,130.00,21.00,6.64
max,2147476221.00,2026-06-10 19:29:39,2026-06-11 03:59:31,2026-08-10 03:21:21,120.00,5061.00,2026.00,100000000.00,41150787.04,214940400.00,105857.00,9651.02,3304.93,27.00,85.00,151.00,43.00,134.00
std,619521183.56,NaN,NaN,NaN,12.89,161.54,4.39,913097.37,172310.99,482566.45,757.21,198.46,858.51,2.67,15.69,31.39,5.65,1.45


In [44]:
clean_kickstarter.to_csv('../data/clean_kickstarter.csv', index=False)